# Imports and Setup

In [ ]:
%load_ext autoreload

import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s') # NOTSET, DEBUG, INFO, WARN, ERROR, CRITICAL

import os
import numpy as np

import jax
import jax.numpy as jnp
from jax import vmap, jit, grad, jacfwd

# Optional: higher numeric accuracy (do this before heavy JAX use)
# try:
#     from jax import config as jax_config
#     jax_config.update("jax_enable_x64", True)
# except Exception:
#     pass

import skewerjax as sjx

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
plt.style.use('tableau-colorblind10')
font, rcnew = sjx.config.setup_matplotlib.matplotlib_default_config()
plt.rc('font', **font)
plt.rcParams.update(rcnew)
%matplotlib widget

# Voigt function

In [ ]:
from skewerjax.optical_depth.voigt import log10_voigt

### Numerical errors and comparison between implementations

In [ ]:
xx = jnp.linspace(-9, 9, 2**13)

sigma_low = 1e-3
sigma_high = 1.0
gamma_low = 1e-3
gamma_high = 1.0
sigma = jnp.array([sigma_low, sigma_high, sigma_low, sigma_high])
gamma = jnp.array([gamma_low, gamma_low, gamma_high, gamma_high])

Voigt_dict = {}

Voigt_dict['scipy_wofz'] = log10_voigt(xx, sigma, gamma, mode="scipy_wofz")

voigt_convolution_batched = vmap(lambda s, g: log10_voigt(xx, s, g, mode="convolution", oversample=4), in_axes=(0, 0))

Voigt_dict["JAX_convolution"] = voigt_convolution_batched(sigma, gamma)

voigt_exojax_batched = vmap(lambda s, g: log10_voigt(xx, s, g, mode="ExoJAX"), in_axes=(0, 0))

Voigt_dict["exojax"] = voigt_exojax_batched(sigma, gamma)

In [ ]:
P = len(sigma)
colors = np.array(plt.cm.get_cmap("tab10").colors[:P])

fig, (ax, ax_res) = plt.subplots(2, 1, figsize=(9, 7), sharex=True, gridspec_kw=dict(height_ratios=[2, 1]))

for ii in range(P):
    ax.plot(xx, 10**Voigt_dict['scipy_wofz'][ii], color=colors[ii], linestyle="-", linewidth=1.0)
    ax.plot(xx, 10**Voigt_dict['JAX_convolution'][ii], color=colors[ii], linestyle="--", linewidth=2.)
    ax.plot(xx, 10**Voigt_dict['exojax'][ii], color=colors[ii], linestyle=":", linewidth=3.5)

for ii in range(P):
    ax_res.plot(
        xx, 10**Voigt_dict['JAX_convolution'][ii] / 10**Voigt_dict['scipy_wofz'][ii] - 1,
        color=colors[ii], linestyle="--", linewidth=1.0
    )
    ax_res.plot(
        xx, 10**Voigt_dict['exojax'][ii] / 10**Voigt_dict['scipy_wofz'][ii] - 1,
        color=colors[ii], linestyle=":", linewidth=2.0
    )

method_handles = [
    Line2D([0], [0], color="k", lw=1,  ls="-",  label="Scipy wofz"),
    Line2D([0], [0], color="k", lw=2,  ls="--", label="JAX convolution"),
    Line2D([0], [0], color="k", lw=3.5,  ls=":",  label="exojax"),
]
leg1 = ax.legend(handles=method_handles, loc="upper right", fontsize=16)
ax.add_artist(leg1)

param_handles = []
for ii in range(P):
    lbl = rf"$\sigma={sigma[ii]:g},\ \gamma={gamma[ii]:g}$"
    param_handles.append(Line2D([0], [0], color=colors[ii], lw=6, label=lbl))

leg2 = ax.legend(handles=param_handles, loc="upper left", fontsize=16)
ax.add_artist(leg2)

ax.set_title("Voigt profiles")
ax.set_ylabel(r"$V(x;\ \sigma,\ \gamma)$")
ax.set_yscale("log")
ax.set_ylim(1e-6, 1e3)

ax_res.axhline(0.0, color="k", linewidth=1, alpha=0.5)
ax_res.set_xlabel("x")        # only bottom panel gets the x-label
ax_res.set_ylabel("residual")
ax_res.set_yscale("symlog", linthresh=1e-6)

major_ticks = [-1e-2, -1e-5, 0.0, 1e-5, 1e-2]
ax_res.set_yticks(major_ticks)

ax_res.yaxis.set_major_formatter(mpl.ticker.FuncFormatter(sjx.utils.plotting_tools.ten_power_formatter))
ax_res.yaxis.set_minor_formatter(mpl.ticker.NullFormatter())

plt.subplots_adjust(hspace=0.05)
fig.tight_layout()
plt.show()

### Autodiff

In [ ]:
xx = jnp.linspace(-9.0, 9.0, 2**12)  # frequency (or velocity) offset grid

sigma0 = 1.0
gamma0 = 0.001

In [ ]:
def voigt_scalar(x, sigma, gamma):
    return 10**log10_voigt(jnp.atleast_1d(x), sigma, gamma, mode="ExoJAX")[0]

# Derivative wrt x (ν)
dVoigt_dx = vmap(grad(voigt_scalar, argnums=0), in_axes=(0, None, None))

# Derivative wrt sigma (Gaussian width)
dVoigt_dsigma = vmap(grad(voigt_scalar, argnums=1), in_axes=(0, None, None))

# Derivative wrt sigma (Lorentzian width)
dVoigt_dgamma = vmap(grad(voigt_scalar, argnums=2), in_axes=(0, None, None))

dV_dx = dVoigt_dx(xx, sigma0, gamma0)
dV_dsigma = dVoigt_dsigma(xx, sigma0, gamma0)
dV_dgamma = dVoigt_dgamma(xx, sigma0, gamma0)

In [ ]:
delta_sigma = 0.2
sigma = jnp.linspace(sigma0 - delta_sigma, sigma0 + delta_sigma, 5)
sigma_ref_idx = len(sigma) // 2
sigma_ref = sigma[sigma_ref_idx]

V_sigma = vmap(lambda s: 10**log10_voigt(xx, s, gamma0), in_axes=(0,))(sigma)
V_sigma_ref = 10**log10_voigt(xx, sigma_ref, gamma0)

delta_gamma = 0.0009
gamma = jnp.linspace(gamma0 - delta_gamma, gamma0 + delta_gamma, 5)
gamma_ref_idx = len(gamma) // 2
gamma_ref = gamma[gamma_ref_idx]

V_gamma = vmap(lambda g: 10**log10_voigt(xx, sigma0, g), in_axes=(0,))(gamma)
V_gamma_ref = 10**log10_voigt(xx, sigma0, gamma_ref)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 6), sharex='col', gridspec_kw=dict(height_ratios=[2, 1]))

ax_main_s, ax_main_g = axes[0, 0], axes[0, 1]
ax_res_s, ax_res_g = axes[1, 0], axes[1, 1]

# =============================================================================
# Sigma variation (left column)
# =============================================================================
colors_sigma = np.array(plt.colormaps["Reds"](np.linspace(0, 1, len(sigma)+2)))[1:-1]

for i in range(len(sigma)):
    ax_main_s.plot(xx, V_sigma[i], color=colors_sigma[i], linewidth=2.0, linestyle='-')

for i in range(len(sigma)):
    residual = V_sigma[i] / V_sigma_ref - 1
    ax_res_s.plot(xx, residual, color=colors_sigma[i], linewidth=2.0, linestyle='-')

# Legend
param_handles_s = []
for i, s_val in enumerate(sigma):
    param_handles_s.append(Line2D([0], [0], color=colors_sigma[i], lw=6, label=rf"$\sigma={s_val:.3g}$"))
leg2_s = ax_main_s.legend(handles=param_handles_s, loc="upper left", fontsize=16)
ax_main_s.add_artist(leg2_s)

# Styling
ax_main_s.set_title(r"Voigt profile variation with $\sigma$", pad=10)
ax_main_s.set_ylabel(r"$V(x;\ \sigma,\ \gamma_0)$")
ax_main_s.set_yscale("log")
V_min_s, V_max_s = V_sigma.min(), V_sigma.max()
ax_main_s.set_ylim(V_min_s * 0.5, V_max_s * 2.0)

ax_res_s.axhline(0.0, color="k", linewidth=1, alpha=0.5)
ax_res_s.set_xlabel("x")
ax_res_s.set_ylabel("residual")
ax_res_s.set_yscale("symlog", linthresh=1e-6)
major_ticks_s = [-1e-2, -1e-5, 0.0, 1e-5, 1e-2]
ax_res_s.set_yticks(major_ticks_s)
ax_res_s.yaxis.set_major_formatter(mpl.ticker.FuncFormatter(sjx.utils.plotting_tools.ten_power_formatter))
ax_res_s.yaxis.set_minor_formatter(mpl.ticker.NullFormatter())

# =============================================================================
# Gamma variation (right column)
# =============================================================================
colors_gamma = np.array(plt.colormaps["Greens"](np.linspace(0, 1, len(gamma)+2)))[1:-1]

for i in range(len(gamma)):
    ax_main_g.plot(xx, V_gamma[i], color=colors_gamma[i], linewidth=2.0, linestyle='-')

for i in range(len(gamma)):
    residual = V_gamma[i] / V_gamma_ref - 1
    ax_res_g.plot(xx, residual, color=colors_gamma[i], linewidth=2.0, linestyle='-')

# Legend
param_handles_g = []
for i, g_val in enumerate(gamma):
    param_handles_g.append(Line2D([0], [0], color=colors_gamma[i], lw=6, label=rf"$\gamma={g_val:.4g}$"))
leg2_g = ax_main_g.legend(handles=param_handles_g, loc="upper left", fontsize=16)
ax_main_g.add_artist(leg2_g)

# Styling
ax_main_g.set_title(r"Voigt profile variation with $\gamma$", pad=10)
ax_main_g.set_ylabel(r"$V(x;\ \sigma_0,\ \gamma)$")
ax_main_g.set_yscale("log")
V_min_g, V_max_g = V_gamma.min(), V_gamma.max()
ax_main_g.set_ylim(V_min_g * 0.5, V_max_g * 2.0)

ax_res_g.axhline(0.0, color="k", linewidth=1, alpha=0.5)
ax_res_g.set_xlabel("x")
ax_res_g.set_ylabel("residual")
ax_res_g.set_yscale("symlog", linthresh=1e-6)
major_ticks_g = [-1e-2, -1e-5, 0.0, 1e-5, 1e-2]
ax_res_g.set_yticks(major_ticks_g)
ax_res_g.yaxis.set_major_formatter(mpl.ticker.FuncFormatter(sjx.utils.plotting_tools.ten_power_formatter))
ax_res_g.yaxis.set_minor_formatter(mpl.ticker.NullFormatter())

# =============================================================================
fig.tight_layout()
fig.subplots_adjust(hspace=0.05, wspace=0.3)  # Add wspace for column separation

plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 5), sharey=True)

ax = axes[0]
ax.plot(xx, dV_dx)
ax.set_xlabel("x")
ax.set_ylabel(r"dV/dx")
ax.set_title("Derivative wrt x")

ax = axes[1]
ax.plot(xx, dV_dsigma)
ax.set_xlabel("x")
ax.set_ylabel(r"dV/dσ")
ax.set_title("Derivative wrt σ")

ax = axes[2]
ax.plot(xx, dV_dgamma)
ax.set_xlabel("x")
ax.set_ylabel(r"dV/dγ")
ax.set_title("Derivative wrt γ")

fig.tight_layout()
plt.show()

# Lines

In [ ]:
from skewerjax.constants.lines import LINES, get_line, LineBundle

### Basic Usage: Fetch a single line by ID

In [ ]:
# Get the Lyman-alpha transition
lya = get_line("HI-LyA")

print(f"ID: {lya.id}")
print(f"Species: {lya.species}")
print(f"Name: {lya.name}")
print(f"Rest wavelength: {lya.lambda0_A:.4f} Å")
print(f"Oscillator strength: {lya.f_osc:.4f}")
print(f"Einstein coefficient: {lya.A21_Hz:.4e} Hz = {lya.A21_GHz:.4f} GHz")
print(f"Mass: {lya.mass_amu:.4f} amu")

### Explore the Global LINES Database

In [ ]:
print(f"Total lines loaded: {len(LINES)}")

for line_id, line in sorted(LINES.items(), key=lambda x: x[1].lambda0_A):
    print(f"  Name:{line_id:11s} λ={line.lambda0_A:8.4f} Å. Species: {line.species:7s} f={line.f_osc:.4f} ")

### Load some selected absorption lines using the LineBundle class

In [ ]:
line_bundle = LineBundle(["HI-LyA", "HI-LyB", "SiIII-1206"])

print(f"Lines: {line_bundle.n_lines}, Species: {line_bundle.n_species}")
print(f"Species keys: {line_bundle.species_keys}")

for ii in range(line_bundle.n_lines):
    print(f"{ii:2d} {line_bundle.lines[ii].id} {line_bundle.lines[ii].species} {line_bundle.lines[ii].lambda0_A} {line_bundle.lines[ii].f_osc} {line_bundle.lines[ii].A21_GHz} {line_bundle.lines[ii].mass_amu}")

print(f"HI lines: {line_bundle.lines_for_species('HI')}")
print(f"SiIII lines: {line_bundle.lines_for_species('SiIII')}")

print(f"HI indices: {line_bundle.indices_for_species('HI')}")
print(f"SiIII indices: {line_bundle.indices_for_species('SiIII')}")

print(f"Lambda0: {line_bundle.lambda0_A}")
print(f"A21: {line_bundle.A21_GHz}")
print(f"Mass: {line_bundle.mass_amu}")

# Fields

In [ ]:
from skewerjax.fields.fields import ScalarField, VectorField, Fields

In [ ]:
def temperature_sampler(T0_min: float = 1e0, T0_max: float = 1e6, L_box: float = 1.0):
    def _sample(x: jnp.ndarray) -> jnp.ndarray:
        # Capture batch dimensions (e.g., (7,5) from (7,5,4))
        *batch_dims, _ = x.shape
        # Flatten all but last dimension: (..., 4) -> (N_points, 4)
        x_flat = x.reshape(-1, 4)
        # Compute on flat array
        z = x_flat[:, -1]  # z-position
        log_T0_min = jnp.log10(T0_min)
        log_T0_max = jnp.log10(T0_max)
        T_flat = 10**(log_T0_min + (log_T0_max - log_T0_min) * (z / L_box))
        # Restore original batch shape
        return T_flat.reshape(batch_dims)
    return _sample


def velocity_sampler(v0_min: float = -0.001, v0_max: float = 0.001, L_box: float = 1.0):
    def _sample(x: jnp.ndarray) -> jnp.ndarray:
        *batch_dims, _ = x.shape
        x_flat = x.reshape(-1, 4)
        # Extract x-position and compute vx
        x_pos = x_flat[:, 1]
        vx_flat = v0_min + (v0_max - v0_min) * x_pos / L_box
        # Build 3D vector: (N_points, 3)
        v_flat = jnp.stack([vx_flat, jnp.zeros_like(vx_flat), jnp.zeros_like(vx_flat)], axis=-1)
        # Restore batch shape + vector dimension: (..., 3)
        return v_flat.reshape(*batch_dims, 3)
    return _sample


def turbulence_sampler(b_turb: float = 0.0):
    def _sample(x: jnp.ndarray) -> jnp.ndarray:
        *batch_dims, _ = x.shape
        x_flat = x.reshape(-1, 4)
        result_flat = jnp.full(x_flat.shape[0], b_turb)
        return result_flat.reshape(batch_dims)
    return _sample


def collisional_sampler(A_coll: float = 0.0):
    def _sample(x: jnp.ndarray) -> jnp.ndarray:
        *batch_dims, _ = x.shape
        x_flat = x.reshape(-1, 4)
        result_flat = jnp.full(x_flat.shape[0], A_coll)
        return result_flat.reshape(batch_dims)
    return _sample


def number_density_sampler(n0_min: float = 1e0, n0_max: float = 1e6, L_box: float = 1.0):
    def _sample(x: jnp.ndarray) -> jnp.ndarray:
        *batch_dims, _ = x.shape
        x_flat = x.reshape(-1, 4)
        y = x_flat[:, 2]  # y-position
        log_n0_min = jnp.log10(n0_min)
        log_n0_max = jnp.log10(n0_max)
        n_flat = 10**(log_n0_min + (log_n0_max - log_n0_min) * (y / L_box))
        return n_flat.reshape(batch_dims)
    return _sample


L_box = 1.0

fields = Fields({
    "HI": {
        "temperature": ScalarField(temperature_sampler(T0_min=1e0, T0_max=1e6, L_box=L_box)),
        "velocity": VectorField(velocity_sampler(v0_min=-0.001, v0_max=0.001, L_box=L_box)),
        "turbulence": ScalarField(turbulence_sampler(b_turb=0.0)),
        "collisional": ScalarField(collisional_sampler(A_coll=0.0)),
        "number_density": ScalarField(number_density_sampler(n0_min=1e0, n0_max=1e6, L_box=L_box)),
    },
    "SiIII": {
        "temperature": ScalarField(temperature_sampler(T0_min=1e0, T0_max=1e6, L_box=L_box)),
        "velocity": VectorField(velocity_sampler(v0_min=-0.001, v0_max=0.001, L_box=L_box)),
        "turbulence": ScalarField(turbulence_sampler(b_turb=0.0)),
        "collisional": ScalarField(collisional_sampler(A_coll=0.0)),
        "number_density": ScalarField(number_density_sampler(n0_min=1e0, n0_max=1e6, L_box=L_box)),
    }
})

In [ ]:
def make_plane_points(L_box, fixed_axis, fixed_value, scale_factor, n_var1, n_var2):
    """Points on plane fixed_axis='x'/'y'/'z'. Shape: (n_var1, n_var2, 4)."""
    # Map: fixed → (fixed_idx, var1_idx, var2_idx)
    f, v1, v2 = {'x': (0,1,2), 'y': (1,2,0), 'z': (2,0,1)}[fixed_axis]
    
    c1 = jnp.linspace(0.0, L_box, n_var1)
    c2 = jnp.linspace(0.0, L_box, n_var2)
    g1, g2 = jnp.meshgrid(c1, c2, indexing="ij")
    
    # Initialize coordinates with dummy arrays, then overwrite
    coords = [jnp.zeros_like(g1) for _ in range(3)]
    coords[f] = jnp.full_like(g1, fixed_value)
    coords[v1] = g1
    coords[v2] = g2
    
    return jnp.stack([jnp.full_like(g1, scale_factor)] + coords, axis=-1)


def make_LoS(batch_shape, direction):
    """Normalized LoS directions: batch_shape + (3,)."""
    base = jnp.asarray(direction, dtype=float)
    loS = jnp.broadcast_to(base, batch_shape + (3,))
    return loS / jnp.linalg.norm(loS, axis=-1, keepdims=True)


scale_factor = 1.0

N_sk = 7
N_ppsk = 5
N_grid = 64

points_1 = make_plane_points(L_box, 'y', L_box/2, scale_factor, N_sk, N_ppsk)
LoS_1 = make_LoS(points_1.shape[:-1], [1.0, 0.0, 0.0])
sampled_fields_1 = fields.sample(points_1, LoS_1)

points_T_imshow = make_plane_points(L_box, 'y', L_box/2, scale_factor, N_grid, N_grid)
T_imshow = fields.temperature(points_T_imshow)

points_2 = make_plane_points(L_box, 'x', L_box/2, scale_factor, N_sk, N_ppsk)
LoS_2 = make_LoS(points_2.shape[:-1], [0.0, 0.0, 1.0])
sampled_fields_2 = fields.sample(points_2, LoS_2)

points_n_imshow = make_plane_points(L_box, 'x', L_box/2, scale_factor, N_grid, N_grid)
n_imshow = fields.number_density(points_n_imshow)

In [ ]:
marker_scatter_1 = "o"
marker_scatter_2 = "s"

key_species = "HI"
i_species = line_bundle.species_keys.index(key_species)

In [ ]:
fig, (axT, axN) = plt.subplots(
    1, 2, figsize=(14, 6), sharey=True,
    gridspec_kw=dict(wspace=0.25)
)

# ============================================================
# LEFT PANEL: Temperature slice (x–z, y = L_box/2)
# ============================================================
# Background: log10 T
logT = np.log10(np.array(T_imshow[i_species]))     # (N_grid, N_grid)
x_min, x_max = float(points_T_imshow[..., 1].min()), float(points_T_imshow[..., 1].max())
z_min_T, z_max_T = float(points_T_imshow[..., -1].min()), float(points_T_imshow[..., -1].max())

norm_T = mpl.colors.Normalize(vmin=logT.min(), vmax=logT.max())
cmap_T = mpl.cm.Reds

im_T = axT.imshow(
    logT,
    origin="lower",
    extent=(x_min, x_max, z_min_T, z_max_T),
    aspect="auto",
    cmap=cmap_T,
    norm=norm_T,
)
cbar_T = fig.colorbar(im_T, ax=axT, pad=0.02, fraction=0.046)
cbar_T.set_label(r"$\log_{10} T\ \mathrm{[K]}$")

# Skewer points colored by log10 T
T_all = np.asarray(sampled_fields_1[key_species]["temperature"])   # (N_sk, N_ppsk)
log10_T_all = np.log10(T_all)
x1 = np.asarray(points_1[..., 1])  # (N_sk, N_ppsk)
z1 = np.asarray(points_1[..., -1]) # (N_sk, N_ppsk)

axT.scatter(
    x1.ravel(), z1.ravel(),
    c=log10_T_all.ravel(),
    cmap=cmap_T,
    norm=norm_T,
    marker=marker_scatter_1,
    s=100,
    alpha=1.0,
    edgecolor="k",
    linewidth=0.5,
)

# Skewer lines + LoS arrows
for i in range(points_1.shape[0]):
    axT.plot(
        points_1[i, ..., 1],
        points_1[i, ..., -1],
        color="k",
        linewidth=1.0,
        alpha=1.0,
    )
    axT.quiver(
        points_1[i, ..., 1],
        points_1[i, ..., -1],
        LoS_1[i, :, 0].T,
        LoS_1[i, :, 2].T,
        color="k",
        angles="xy",
        scale_units="xy",
        scale=None,
        width=0.005,
        alpha=1.0,
    )

# Gas velocities (xz components)
axT.quiver(
    points_1[..., 1], points_1[..., -1],
    sampled_fields_1[key_species]["velocity"][..., 0],
    sampled_fields_1[key_species]["velocity"][..., 2],
    color="royalblue",
    angles="xy",
    scale_units="xy",
    scale=None,
    width=0.007,
    alpha=0.9,
)

# Domain frame & limits
axT.plot(
    [x_min, x_max, x_max, x_min, x_min],
    [z_min_T, z_min_T, z_max_T, z_max_T, z_min_T],
    color="k",
    linewidth=2.5,
    alpha=0.9,
)
limit_padding = 0.2
axT.set_xlim(x_min - limit_padding, x_max + limit_padding)
axT.set_ylim(z_min_T - limit_padding, z_max_T + limit_padding)

# Labels, title, grid
axT.set_xlabel(r"$x$")
axT.set_ylabel(r"$z$")
axT.set_title(rf"{key_species} $T$ slice at $y={float(L_box/2):.3g}$", pad=12)
axT.grid(color="w", linewidth=0.8, alpha=0.15)

# Label each skewer with its (log10) temperature
T_sample = np.array(sampled_fields_1[key_species]["temperature"])  # (N_sk, N_ppsk)
dz_T = (z_max_T - z_min_T)
for i in range(points_1.shape[0]):
    z_i = float(points_1[i, 0, -1])
    x_center = float(points_1[i, ..., 1].mean())
    logT_i = np.log10(T_sample[i, 0])  # assume constant along skewer
    axT.text(
        x_center,
        z_i + 0.02 * dz_T,
        rf"$\log_{{10}} T = {logT_i:.2f}$",
        color="k",
        ha="center",
        va="bottom",
        fontsize=11,
        bbox=dict(boxstyle="round,pad=0.2", facecolor="k", alpha=0.0),
    )

# Legend for left panel
legend_handles_T = [
    Line2D([0], [0], marker="o", linestyle="none",
           markerfacecolor="red", markeredgecolor="k",
           markersize=10, label="sampling points"),
    Line2D([0], [0], color="k", lw=2, label="LoS direction"),
    Line2D([0], [0], color="royalblue", lw=2, label=r"$\vec{v}$"),
]
leg_T = axT.legend(
    handles=legend_handles_T,
    loc="lower center",
    frameon=True,
    fontsize=12,
    framealpha=0.9,
    ncols=3,
)
leg_T.get_frame().set_edgecolor("k")

# ============================================================
# RIGHT PANEL: Number-density slice (y–z, x = L_box/2)
# ============================================================
# Background: log10 n
logn = np.log10(np.array(n_imshow[i_species]))     # (N_grid, N_grid)
y_min, y_max = float(points_n_imshow[..., 2].min()), float(points_n_imshow[..., 2].max())
z_min_n, z_max_n = float(points_n_imshow[..., 3].min()), float(points_n_imshow[..., 3].max())

norm_n = mpl.colors.Normalize(vmin=logn.min(), vmax=logn.max())
cmap_n = mpl.cm.Greens

im_n = axN.imshow(
    logn.T,
    origin="lower",
    extent=(y_min, y_max, z_min_n, z_max_n),
    aspect="auto",
    cmap=cmap_n,
    norm=norm_n,
)
cbar_n = fig.colorbar(im_n, ax=axN, pad=0.02, fraction=0.046)
cbar_n.set_label(r"$\log_{10} n\ \mathrm{[nm^{-3}]}$")

# Skewer points colored by log10 n
n_all = np.asarray(sampled_fields_2[key_species]["number_density"])  # (N_sk, N_ppsk)
log10_n_all = np.log10(n_all)
y2 = np.asarray(points_2[..., 2])  # (N_sk, N_ppsk)
z2 = np.asarray(points_2[..., 3])  # (N_sk, N_ppsk)

axN.scatter(
    y2.ravel(), z2.ravel(),
    c=log10_n_all.ravel(),
    cmap=cmap_n,
    norm=norm_n,
    marker=marker_scatter_2,
    s=100,
    alpha=1.0,
    edgecolor="k",
    linewidth=0.5,
)

# Skewer lines + LoS arrows
for i in range(points_2.shape[0]):
    axN.plot(
        points_2[i, ..., 2],
        points_2[i, ..., 3],
        color="k",
        linewidth=1.0,
        alpha=1.0,
    )
    axN.quiver(
        points_2[i, ..., 2],
        points_2[i, ..., 3],
        LoS_2[i, :, 1].T,
        LoS_2[i, :, 2].T,
        color="k",
        angles="xy",
        scale_units="xy",
        scale=None,
        width=0.005,
        alpha=1.0,
    )

# Gas velocities (yz components)
axN.quiver(
    points_2[..., 2], points_2[..., 3],
    sampled_fields_2[key_species]["velocity"][..., 1],
    sampled_fields_2[key_species]["velocity"][..., 2],
    color="royalblue",
    angles="xy",
    scale_units="xy",
    scale=None,
    width=0.007,
    alpha=0.9,
)

# Domain frame & limits (use y,z extents here)
axN.plot(
    [y_min, y_max, y_max, y_min, y_min],
    [z_min_n, z_min_n, z_max_n, z_max_n, z_min_n],
    color="k",
    linewidth=2.5,
    alpha=0.9,
)
axN.set_xlim(y_min - limit_padding, y_max + limit_padding)
axN.set_ylim(z_min_n - limit_padding, z_max_n + limit_padding)

# Labels, title, grid
axN.set_xlabel(r"$y$")
axN.set_ylabel(r"$z$")
axN.set_title(rf"{key_species} $n$ slice at $x={float(L_box/2):.3g}$", pad=12)
axN.grid(color="w", linewidth=0.8, alpha=0.15)

# Legend for right panel
legend_handles_n = [
    Line2D([0], [0], marker="o", linestyle="none",
           markerfacecolor="green", markeredgecolor="k",
           markersize=10, label="sampling points"),
    Line2D([0], [0], color="k", lw=2, label="LoS direction"),
    Line2D([0], [0], color="royalblue", lw=2, label=r"$\vec{v}$"),
]
leg_n = axN.legend(
    handles=legend_handles_n,
    loc="lower center",
    frameon=True,
    fontsize=12,
    framealpha=0.9,
    ncols=3,
)
leg_n.get_frame().set_edgecolor("k")

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

In [ ]:
L = float(L_box)

# ----------------------
# Y-plane (temperature)
# ----------------------
# coordinates
x_y = np.asarray(points_1[..., 1])   # (N_sk, N_ppsk)
y_y = np.asarray(points_1[..., 2])   # constant y = y_plane
z_y = np.asarray(points_1[..., 3])   # (N_sk, N_ppsk)

y_plane = float(y_y[0, 0])

# sampled fields
T_y   = np.asarray(sampled_fields_1[key_species]["temperature"])   # (N_sk, N_ppsk)
vel_y = np.asarray(sampled_fields_1[key_species]["velocity"])      # (N_sk, N_ppsk, 3)
LoS_y = np.asarray(LoS_1)                                          # (N_sk, N_ppsk, 3)

logT_y = np.log10(T_y)
norm_T = mpl.colors.Normalize(vmin=logT_y.min(), vmax=logT_y.max())
cmap_T = mpl.cm.Reds

# ----------------------
# X-plane (number density) – single plane at x = L_box/2
# ----------------------
x_x = np.asarray(points_2[..., 1])    # constant x = x_plane
y_x = np.asarray(points_2[..., 2])    # (N_sk, N_ppsk)
z_x = np.asarray(points_2[..., 3])    # (N_sk, N_ppsk)

x_plane = float(x_x[0, 0])

n_all = np.asarray(sampled_fields_2[key_species]["number_density"])  # (N_sk, N_ppsk)
vel_x = np.asarray(sampled_fields_2[key_species]["velocity"])        # (N_sk, N_ppsk, 3)
LoS_x = np.asarray(LoS_2)                                            # (N_sk, N_ppsk, 3)

logn_all = np.log10(n_all)
norm_n = mpl.colors.Normalize(vmin=logn_all.min(), vmax=logn_all.max())
cmap_n = mpl.cm.Greens

# ---------------------------------------------------------
# 3D plot
# ---------------------------------------------------------
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection="3d")

# Some style choices (you probably already have these defined)
los_line_color  = "k"
vel_arrow_color = "tab:blue"

# =========================================================
# Y-plane: points + lines + arrows
# =========================================================
N_sk, N_ppsk = x_y.shape

# Flatten for scatter
x_y_f    = x_y.ravel()
y_y_f    = y_y.ravel()
z_y_f    = z_y.ravel()
logT_y_f = logT_y.ravel()
colors_y = cmap_T(norm_T(logT_y_f))

ax.scatter(
    x_y_f, y_y_f, z_y_f,
    c=colors_y,
    marker="o",
    s=100,
    alpha=0.9,
    edgecolor="k",
    linewidth=0.5,
)

# Join points along each skewer (fixed z index)
for i in range(N_sk):
    ax.plot(
        x_y[i, :], y_y[i, :], z_y[i, :],
        color=los_line_color,
        linewidth=1.0,
        alpha=0.9,
    )

# Subsample for arrows
N_sk_y, N_ppsk_y, _ = vel_y.shape
step_z_y = max(1, N_sk_y  // 5)
step_x_y = max(1, N_ppsk_y // 5)

Xq_y = x_y[::step_z_y, ::step_x_y]
Yq_y = y_y[::step_z_y, ::step_x_y]
Zq_y = z_y[::step_z_y, ::step_x_y]

U_LoS_y = LoS_y[::step_z_y, ::step_x_y, 0]
V_LoS_y = LoS_y[::step_z_y, ::step_x_y, 1]
W_LoS_y = LoS_y[::step_z_y, ::step_x_y, 2]

U_v_y = vel_y[::step_z_y, ::step_x_y, 0]
V_v_y = vel_y[::step_z_y, ::step_x_y, 1]
W_v_y = vel_y[::step_z_y, ::step_x_y, 2]

# LoS arrows (y-plane) – all black
ax.quiver(
    Xq_y, Yq_y, Zq_y,
    U_LoS_y, V_LoS_y, W_LoS_y,
    length=0.07 * L,
    normalize=True,
    color=los_line_color,
    linewidth=1.0,
    alpha=0.9,
)

# 3D velocity arrows (y-plane) – all same color
ax.quiver(
    Xq_y, Yq_y, Zq_y,
    U_v_y, V_v_y, W_v_y,
    length=0.07 * L,
    normalize=True,
    color=vel_arrow_color,
    linewidth=1.5,
    alpha=0.9,
)

# =========================================================
# X-plane: points + lines + arrows (using sampled_fields_2)
# =========================================================
# You can pick any marker; here just one marker type
x_plane_marker = "^"

ax.scatter(
    x_x.ravel(), y_x.ravel(), z_x.ravel(),
    c=cmap_n(norm_n(logn_all.ravel())),
    marker=x_plane_marker,
    s=100,
    alpha=0.9,
    edgecolor="k",
    linewidth=0.5,
)

# Join points along skewers (fixed "z index" – i.e., along y axis)
for i in range(N_sk):
    ax.plot(
        x_x[i, :], y_x[i, :], z_x[i, :],
        color=los_line_color,
        linewidth=1.0,
        alpha=0.9,
    )

# Subsample for arrows
N_sk_x, N_ppsk_x, _ = vel_x.shape
step_z_x = max(1, N_sk_x  // 5)
step_y_x = max(1, N_ppsk_x // 5)

Xq_x = x_x[::step_z_x, ::step_y_x]
Yq_x = y_x[::step_z_x, ::step_y_x]
Zq_x = z_x[::step_z_x, ::step_y_x]

U_LoS_x = LoS_x[::step_z_x, ::step_y_x, 0]
V_LoS_x = LoS_x[::step_z_x, ::step_y_x, 1]
W_LoS_x = LoS_x[::step_z_x, ::step_y_x, 2]

U_v_x = vel_x[::step_z_x, ::step_y_x, 0]
V_v_x = vel_x[::step_z_x, ::step_y_x, 1]
W_v_x = vel_x[::step_z_x, ::step_y_x, 2]

# LoS arrows (x-plane) – all black
ax.quiver(
    Xq_x, Yq_x, Zq_x,
    U_LoS_x, V_LoS_x, W_LoS_x,
    length=0.07 * L,
    normalize=True,
    color=los_line_color,
    linewidth=1.0,
    alpha=0.9,
)

# 3D velocity arrows (x-plane) – same color as y-plane
ax.quiver(
    Xq_x, Yq_x, Zq_x,
    U_v_x, V_v_x, W_v_x,
    length=0.07 * L,
    normalize=True,
    color=vel_arrow_color,
    linewidth=1.5,
    alpha=0.9,
)

# =========================================================
# Intersection line of x-plane and y-plane
# =========================================================
z_line = np.linspace(0.0, L, 100)
x_line = np.full_like(z_line, x_plane)
y_line = np.full_like(z_line, y_plane)
ax.plot(
    x_line, y_line, z_line,
    color="k",
    linewidth=1.5,
    alpha=0.5,
)

# =========================================================
# Colorbars for T and n (using precomputed logs)
# =========================================================
sm_T = mpl.cm.ScalarMappable(norm=norm_T, cmap=cmap_T)
sm_T.set_array([])
sm_n = mpl.cm.ScalarMappable(norm=norm_n, cmap=cmap_n)
sm_n.set_array([])

# Place them manually on the right
cax_T = fig.add_axes([0.88, 0.15, 0.02, 0.30])  # lower bar
cax_n = fig.add_axes([0.88, 0.55, 0.02, 0.30])  # upper bar

cbar_T = fig.colorbar(sm_T, cax=cax_T)
cbar_T.set_label(r"$\log_{10} T\ \mathrm{[K]}$")

cbar_n = fig.colorbar(sm_n, cax=cax_n)
cbar_n.set_label(r"$\log_{10} n\ \mathrm{[nm^{-3}]}$")

# =========================================================
# Axes, aspect, limits, ticks, legend
# =========================================================
ax.set_xlabel(r"$x$")
ax.set_ylabel(r"$y$")
ax.set_zlabel(r"$z$")
ax.set_title(rf"{key_species} 3D skewers", pad=12)

ax.grid(False)

pad = 0.05 * L
ax.set_xlim(-pad, L + pad)
ax.set_ylim(-pad, L + pad)
ax.set_zlim(-pad, L + pad)

ax.xaxis.set_major_locator(mpl.ticker.MaxNLocator(4))
ax.yaxis.set_major_locator(mpl.ticker.MaxNLocator(4))
ax.zaxis.set_major_locator(mpl.ticker.MaxNLocator(4))

# remove axis panes (background planes)
for axis in (ax.xaxis, ax.yaxis, ax.zaxis):
    try:
        axis.pane.set_visible(False)
    except AttributeError:
        pass

ax.set_box_aspect((L + 2*pad, L + 2*pad, L + 2*pad))
ax.view_init(elev=28, azim=-55)

legend_handles = [
    Line2D(
        [0], [0], marker="o", linestyle="none",
        markerfacecolor="0.7", markeredgecolor="k",
        markersize=8, label=rf"y-plane, $y={y_plane:.3g}$ (T)",
    ),
    Line2D(
        [0], [0], marker="^", linestyle="none",
        markerfacecolor="0.7", markeredgecolor="k",
        markersize=8, label=rf"x-plane, $x={x_plane:.3g}$ (n)",
    ),
]

leg = ax.legend(
    handles=legend_handles,
    loc="upper left",
    fontsize=16,
    frameon=True,
    framealpha=0.9,
)
leg.get_frame().set_edgecolor("k")

plt.tight_layout()
plt.show()


# Line Profiles

In [ ]:
from skewerjax.optical_depth.optical_depth import compute_line_profiles

In [ ]:
wavelengths = jnp.linspace(970, 1280, 2**18)  # in Angstroms

points = points_1
LoS_directions = LoS_1

In [ ]:
phi_ns = compute_line_profiles(
    wavelengths,
    points,
    LoS_directions,
    line_bundle,
    fields,
    compute_mode="pmap",
    voigt_implementation_mode = "ExoJAX",
    num_gpus = None
)

In [ ]:
# Convert JAX arrays to NumPy for plotting
lambdas_np = np.asarray(wavelengths)           # (N_lambda,)
phi_np = np.asarray(phi_ns)                     # (N_sp, N_sk, N_ppsk, N_lambda)

colors = np.array(plt.cm.get_cmap("tab10").colors[:N_sk])
linestyles = ['-', '--', ':']

i_ppsk = N_ppsk // 2

fig, ax = plt.subplots(figsize=(14, 6))

# Plot all profiles on the same axes
for i_sp, line in enumerate(line_bundle.lines):
    ax.axvline(line.lambda0_A, color="grey", linestyle="-.", linewidth=1.5, alpha=1.0)
    ax.scatter(np.array([line.lambda0_A]), np.array([4./line.A21_GHz]), color="grey", s=50, alpha=1.0)
    ls = linestyles[i_sp % len(linestyles)]
    for i_sk in range(N_sk):
        profile = phi_np[i_sp, i_sk, i_ppsk, :]   # full wavelength range
        ax.plot(lambdas_np, profile, color=colors[i_sk], linestyle=ls, linewidth=2.0)

# Axes formatting
ax.set_yscale("log")
ax.set_xlabel(r"Wavelength $\lambda\ [\mathrm{\AA}]$")
ax.set_ylabel(r"$\phi(\lambda)\; \left[\mathrm{ns}\right]$")
ax.grid(alpha=0.2, which="both")
ax.set_title("Line profiles for different transitions and temperatures", fontsize=20)

# Legend for temperatures (colors)
temp_handles = []
for i_sk in range(N_sk):
    _sampled_T = fields.temperature(points[i_sk])
    print(_sampled_T.shape)
    lbl = rf"$T = {_sampled_T[0, 0]:.2g}\,\mathrm{{K}}$"
    temp_handles.append(Line2D([0], [0], color=colors[i_sk], lw=4, linestyle="-", label=lbl))
leg_T = ax.legend(handles=temp_handles, loc="upper left", fontsize=16, ncols=1)
ax.add_artist(leg_T)


# Legend for lines (linestyles)
line_handles = []
for i_sp, line in enumerate(line_bundle.lines):
    ls = linestyles[i_sp % len(linestyles)]
    lbl = rf"{line.name},  $\lambda_0 = {line.lambda0_A:.3f}\,\mathrm{{\AA}}$"
    line_handles.append(Line2D([0], [0], color="k", lw=3, linestyle=ls, label=lbl))
leg_L = ax.legend(handles=line_handles, loc="lower right", fontsize=16, ncol=line_bundle.n_lines)

fig.tight_layout()
plt.show()



# Zoom half-width around each line center [Angstrom]
delta_lambda_A = 1.3   # adjust to taste
fig, axes = plt.subplots(nrows=1, ncols=line_bundle.n_lines, figsize=(14, 6), sharex=False, sharey=True)
if line_bundle.n_lines == 1:
    axes = [axes]  # in case N_sp=1, make it iterable

for i_sp, (ax, line) in enumerate(zip(axes, line_bundle.lines)):
    lambda0 = line.lambda0_A
    ax.axvline(lambda0, color="grey", linestyle="-.", linewidth=1.5)

    mask = (lambdas_np >= lambda0 - delta_lambda_A) & (lambdas_np <= lambda0 + delta_lambda_A)
    lam_win = lambdas_np[mask]  # (N_lambda_window,)

    # Plot each temperature (each skewer)
    for i_sk in range(N_sk):
        profile = phi_np[i_sp, i_sk, i_ppsk, mask]  # (N_lambda_window,)
        ax.plot(lam_win, profile, color=colors[i_sk], linestyle=linestyles[i_sp], linewidth=2.0)

    # Panel title and scaling
    ax.set_title(rf"{line.name},  $\lambda_0 = {line.lambda0_A:.3f}\,\mathrm{{\AA}}$", fontsize=20)
    ax.set_yscale("log")
    ax.set_xlabel(r"Wavelength $\lambda\ [\mathrm{\AA}]$")
    ax.grid(alpha=0.2, which="both")
axes[0].set_ylabel(r"$\phi(\lambda)\; \left[\mathrm{ns}\right]$")

fig.tight_layout()
plt.show()

In [ ]:
# Convert JAX arrays to NumPy for plotting
lambdas_np = np.asarray(wavelengths)                          # (N_lambda,)
phi_np = np.asarray(phi_ns)                                    # (N_sp, N_sk, N_ppsk, N_lambda)

colors = np.array(plt.cm.get_cmap("tab10").colors[:N_sk])
linestyles = ['-', '--', ':']

i_sk = N_sk // 2

fig, ax = plt.subplots(figsize=(14, 6))

# Plot all profiles on the same axes
for i_sp, line in enumerate(line_bundle.lines):
    ax.axvline(line.lambda0_A, color="grey", linestyle="-.", linewidth=1.5, alpha=1.0)
    ax.scatter(np.array([line.lambda0_A]), np.array([4./line.A21_GHz]), color="grey", s=50, alpha=1.0)
    ls = linestyles[i_sp % len(linestyles)]
    for i_ppsk in range(N_ppsk):
        profile = phi_np[i_sp, i_sk, i_ppsk, :]   # full wavelength range
        ax.plot(lambdas_np, profile, color=colors[i_ppsk], linestyle=ls, linewidth=2.0)

# Axes formatting
ax.set_yscale("log")
ax.set_xlabel(r"Wavelength $\lambda\ [\mathrm{\AA}]$")
ax.set_ylabel(r"$\phi(\lambda)\; \left[\mathrm{ns}\right]$")
ax.grid(alpha=0.2, which="both")
ax.set_title("Line profiles for different transitions and temperatures", fontsize=20)

# Legend for velocities (colors)
temp_handles = []
for i_ppsk in range(N_ppsk):
    _sampled_fields = fields.sample(points[:, i_ppsk], LoS_directions[:, i_ppsk])
    _sampled_vLoS = _sampled_fields['HI']["LoS_velocity"][0]
    lbl = rf"$v_\mathrm{{LOS}}/c = {_sampled_vLoS:.2g}$"
    temp_handles.append(Line2D([0], [0], color=colors[i_ppsk], lw=4, linestyle="-", label=lbl))
leg_T = ax.legend(handles=temp_handles, loc="upper left", fontsize=16, ncols=1)
ax.add_artist(leg_T)



# Legend for lines (linestyles)
line_handles = []
for i_sp, line in enumerate(line_bundle.lines):
    ls = linestyles[i_sp % len(linestyles)]
    lbl = rf"{line.name},  $\lambda_0 = {line.lambda0_A:.3f}\,\mathrm{{\AA}}$"
    line_handles.append(Line2D([0], [0], color="k", lw=3, linestyle=ls, label=lbl))
leg_L = ax.legend(handles=line_handles, loc="lower right", fontsize=16, ncol=line_bundle.n_lines)

fig.tight_layout()
plt.show()


# Zoom half-width around each line center [Angstrom]
delta_lambda_A = 1.3   # adjust to taste
fig, axes = plt.subplots(nrows=1, ncols=line_bundle.n_lines, figsize=(14, 6), sharex=False, sharey=True)
if line_bundle.n_lines == 1:
    axes = [axes]  # in case N_sp=1, make it iterable

for i_sp, (ax, line) in enumerate(zip(axes, line_bundle.lines)):
    lambda0 = line.lambda0_A
    ax.axvline(lambda0, color="grey", linestyle="-.", linewidth=1.5)

    mask = (lambdas_np >= lambda0 - delta_lambda_A) & (lambdas_np <= lambda0 + delta_lambda_A)
    lam_win = lambdas_np[mask]  # (N_lambda_window,)

    # Plot each temperature (each skewer)
    for i_ppsk in range(N_ppsk):
        profile = phi_np[i_sp, i_sk, i_ppsk, mask]  # (N_lambda_window,)
        ax.plot(lam_win, profile, color=colors[i_ppsk], linestyle=linestyles[i_sp], linewidth=2.0)

    # Panel title and scaling
    ax.set_title(rf"{line.name},  $\lambda_0 = {line.lambda0_A:.3f}\,\mathrm{{\AA}}$", fontsize=20)
    ax.set_yscale("log")
    ax.set_xlabel(r"Wavelength $\lambda\ [\mathrm{\AA}]$")
    ax.grid(alpha=0.2, which="both")
axes[0].set_ylabel(r"$\phi(\lambda)\; \left[\mathrm{ns}\right]$")

fig.tight_layout()
plt.show()


# Optical Depths

In [ ]:
# cte_nm2_per_ns = (phys.qe**2 / (4 * phys.epsilon_0 * phys.me * phys.c))*1e3                   # (N_sp, N_sk, N_ppsk, N_lambda) — UNITS: [nm² / ns]
# sigma_nm2 = cte_nm2_per_ns * f_osc[:, None, None, None] * phi_ns                              # (N_sp, N_sk, N_ppsk, N_lambda) — UNITS: [nm²]
# integrand = sigma_nm2 * number_density_inv_nm3[..., None] * scale_factor[None, ..., None]**-2 # (N_sp, N_sk, N_ppsk, N_lambda) — UNITS: [1/nm]

# print("integrand", integrand.shape, integrand.min(), integrand.max())

# Transmitted Fluxes

# tests to explore vmap, pmap and Pytrees

In [ ]:
from jax import vmap, jit, pmap
import jax.numpy as jnp
import matplotlib.pyplot as plt



def make_chi2_computer(model_fn):
    """Factory: model_fn is baked into the returned function."""
    def chi2(data, params):
        model = model_fn(data["x"], params)
        return jnp.sum((data["y"] - model)**2 / data["noise"]**2)
    return chi2

def sinusoid_model(x, params):
    return params["amp"] * jnp.sin(x + params["phase"])


chi2_sinusoid = make_chi2_computer(sinusoid_model)



# Each sightline is a dict of arrays shape (n_points,)
datasets = {
    "x": jnp.array([
        [0.0, 0.5, 1.0, 1.5, 2.0],  # Sightline 0
        [0.0, 0.7, 1.4, 2.1, 2.8],  # Sightline 1
        [0.0, 1.0, 2.0, 3.0, 4.0],  # Sightline 2
    ]),
    "y": jnp.array([
        [0.1, 0.9, 0.2, -0.7, -0.3],  # Observed flux for sightline 0
        [0.0, 0.5, 0.8, 0.3, -0.5],   # Sightline 1
        [0.1, 1.2, 0.9, 0.4, 0.1],    # Sightline 2
    ]),
    "noise": jnp.full((3, 5), 0.05)  # Constant noise for all
}

params_sine = {"amp": 1.0, "phase": 0.0}




vcompute_sinusoid = vmap(chi2_sinusoid, in_axes=(0, None))
chi2_vmap = vcompute_sinusoid(datasets, params_sine)
print(chi2_vmap)

In [ ]:
import jax

In [ ]:
# Simulate 8 GPUs (use real number if available)
n_devices = min(8, jax.local_device_count())
print(f"Available devices: {jax.local_device_count()}, Using: {n_devices}")

# Reshape data: (n_devices, n_sightlines_per_device, ...)
# Pad to multiple of n_devices if needed
n_total = datasets["x"].shape[0]
if n_total % n_devices != 0:
    pad_size = n_devices - (n_total % n_devices)
    datasets = {k: jnp.pad(v, ((0, pad_size), (0, 0))) for k, v in datasets.items()}

datasets_sharded = {
    k: jnp.reshape(v, (n_devices, -1, v.shape[-1])) 
    for k, v in datasets.items()
}
print(f"Sharded shape: {datasets_sharded['x'].shape}")  # (8, 1, 5) if 8 devices

# Create pmapped pipeline
# pmap the vmapped function: one per device
pcompute_sinusoid = pmap(
    vcompute_sinusoid,  # Re-use the vmapped version
    in_axes=(0, None),  # Shard the dataset axis 0, broadcast params
    axis_name="devices"
)

# Run on all devices
chi2_pmap = pcompute_sinusoid(datasets_sharded, params_sine)
print(f"PMAP χ² (sharded): {chi2_pmap}")  # shape: (8, 1)
print(f"PMAP χ² (flattened): {chi2_pmap.flatten()[:n_total]}")